# permbench v0.4 (Kaggle): tolerant object IDs, fair baselines, full test set

**Why this notebook exists.** Inspecting your saved logs showed that most "nonexistent object" calls were ID *formatting* slips (`t20000/d1`, `doc:d1`, a copied search line) that the old resolver turned into missing objects. That charged formatting mistakes to the base-model arms and made the old "real permission attempt" rates for A/B arms (about 1%) meaningless. v0.4 resolves these ids identically for every arm (gateway, labeler, intent matcher, tool backends). Cross-tenant ids stay denied; placeholders such as `<doc-id>` still fail.

**What runs (two T4s in parallel, one process per GPU):**
- GPU 0: `C0` (prompt-only base model) on all 1,468 test episodes, then `A-Ctx` (object list, no permissions, no gateway) on the first 400.
- GPU 1: `B-Ctx-Feedback` and `B-Ctx-Stop` on all 1,468.
- `C` (your LoRA) is **not rerun**: re-labelling its 2,741 logged calls with the new code changes nothing and its stored episode scores are identical (the notebook checks this). So **no adapter and no retraining are needed**.
- Old v0.3 results are kept in `legacy/` as the "strict-format" sensitivity analysis; they are not mixed into the new table.

**Setup (right panel):** GPU on (T4 x2), Internet On, Secret `HF_TOKEN` (only for the optional backup). Run cells in order. A 24-episode smoke test runs first, because the last two crashes were found only after long runs.

**Time:** my estimate is 2 to 3 hours wall-clock (your measured speeds were 1.7 to 5.8 s per episode per arm, so the range is wide). Set `EVAL_N` in the config cell to a number such as 600 to cap it.

Results stay preliminary until the final table is printed: one seed, templated language, Qwen2.5-3B, closed-world intent oracle.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# ---- CONFIG ----
MODEL   = "unsloth/Qwen2.5-3B-Instruct"   # same base model the adapter was trained from
HF_REPO = ""                              # optional private HF repo for a results backup
EVAL_BATCH = 8
EVAL_N = None                             # None = all 1,468 test episodes; or e.g. 600 to cap time (state this in any write-up)
EXPECTED_TEST_SHA = "4d80fca654a4f30087ced8ea26662ea0172b555b27d978a77a04ba59111d59dc"


In [ ]:
import os, sys, subprocess, re, json, time, shutil, glob, io, base64, zipfile, hashlib
ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/kaggle_sim2"
os.makedirs(f"{ROOT}/logs", exist_ok=True)
def run(cmd, log, check=True):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    out, n_prog = [], 0
    print(f"$ {cmd}\n", flush=True)
    with open(f"{ROOT}/logs/{log}.log", "w") as f:
        p = subprocess.Popen(cmd, shell=True, cwd=os.getcwd(), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            f.write(line); out.append(line)
            if re.search(r"\d+%\|", line) or "max_new_tokens" in line:   # thin out progress bars and the repeated generation warning
                n_prog += 1
                if n_prog % 40: continue
            sys.stdout.write(line); sys.stdout.flush()
        p.wait()
    if check and p.returncode: raise RuntimeError(f"command failed (exit {p.returncode}); log: {ROOT}/logs/{log}.log")
    return "".join(out)
q = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
print(q.stdout or "NO GPU: enable an accelerator in Session options"); print("ROOT =", ROOT)

In [ ]:
# ---- unpack the v0.4 project and the v0.3 results (both embedded) ----
PROJ_B64 = """
...
AA0A8gIAACeAAAAAAA==
"""
RES_B64 = """
...
AA==
"""
pz = base64.b64decode("".join(PROJ_B64.split())); rz = base64.b64decode("".join(RES_B64.split()))
assert hashlib.sha256(pz).hexdigest() == "f62de1778ad6dd0f25fd9e9138bd0fc0d0ea6650c637572fd4571bad7c9072cb" and hashlib.sha256(rz).hexdigest() == "48cf7e402226f613326cb327b97b82931f94a4cff1d6a64513bc9fbf0bd90d13", "embedded data corrupted"
PROJ = f"{ROOT}/permbench"; os.makedirs(PROJ, exist_ok=True)
zipfile.ZipFile(io.BytesIO(pz)).extractall(PROJ); os.chdir(PROJ)
zipfile.ZipFile(io.BytesIO(rz)).extractall(".")            # -> legacy/*.json (v0.3 results, strict-format)
print(run("python patch_v04.py", "patch"))                 # idempotent: the embedded code is already patched
os.makedirs("results", exist_ok=True)
for f in ("lora_C.json", "lora_rest_C.json"): shutil.copy(f"legacy/{f}", f"results/{f}")   # C is valid under v0.4 (checked below)
print("legacy:", sorted(os.listdir("legacy"))); print("results:", sorted(os.listdir("results")))


In [ ]:
# ---- install (as before) ----
run("python -m pip install -q -U pip && python -m pip install -q -U unsloth", "install1")
run('python -m pip install -q "transformers>=5.2.0,<=5.17.0" tqdm requests huggingface_hub', "install2")
print(run("python -m pip check || true", "pipcheck", check=False))

In [ ]:
# ---- regenerate the frozen test set (deterministic) and check it is identical ----
txt = run("python gen_data.py", "gen_data")
m = re.search(r"test_episodes.json sha256: ([0-9a-f]{64})", txt)
assert "self-checks passed" in txt and m and m.group(1) == EXPECTED_TEST_SHA, "test set differs from the frozen one: STOP"
print("GATE OK: frozen test set (v0.3 hash; same episodes as the first run, intent field corrected)")

In [ ]:
# ---- GATE: the new ID resolver must change nothing except R_NOT_FOUND labels; LoRA results must be untouched ----
out = run("python relabel_check.py", "relabel_check")
assert "UNEXPECTED CHANGES (a label other than R_NOT_FOUND changed): 0" in out, "resolver changed a label it should not have: STOP"
assert out.count("stored episode scores identical: True") == 2, "LoRA episode scores changed: STOP"
print("GATE OK: C (LoRA) results are valid under v0.4 and need no rerun")


In [ ]:
# ---- SMOKE: 24 episodes, all new arm types, both GPUs idle otherwise (3-6 min). Catches crashes before the long run ----
lim = 24
run(f"python eval_models.py --model {MODEL} --setups A-Ctx B-Ctx-Feedback B-Ctx-Stop C0 --tag smoke_v04 --limit {lim} --batch {EVAL_BATCH}", "smoke")
for f in sorted(glob.glob("results/smoke_v04_*.json")):
    r = json.load(open(f)); assert json.dumps(r["logs"]).count("<error") == 0, f"silent generation errors in {f}"
    print(os.path.basename(f), "n =", r["n"], "| seconds/episode = %.1f" % (r["seconds"] / r["n"]), "| TaskSuccess = %.2f" % r["overall"]["TaskSuccess"])
print("SMOKE OK (smoke files are ignored by rescore.py)")


In [ ]:
# ---- PARALLEL RUN on both T4s: one independent process per GPU ----
import subprocess, time, re, os
def launch(gpu, cmds, name):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    f = open(f"{ROOT}/logs/{name}.log", "w")
    return subprocess.Popen(" && ".join(cmds), shell=True, cwd=os.getcwd(), env=env, stdout=f, stderr=subprocess.STDOUT)
def last_progress(name):
    try: t = open(f"{ROOT}/logs/{name}.log", errors="ignore").read()[-8000:]
    except Exception: return "(no log)"
    ms = [m.group(0) for m in re.finditer(r"[A-Za-z0-9-]+:\s+\d+%\|[^\r\n]*", t)]
    return ms[-1][:110] if ms else "(loading / starting)"
LIM = f"--limit {EVAL_N}" if EVAL_N else ""
LIM_A = f"--limit {min(400, EVAL_N or 400)}"
E = f"python eval_models.py --batch {EVAL_BATCH}"
Q0 = [f"{E} --model {MODEL} --setups C0 --tag v04_C0 {LIM}",              # GPU 0: prompt-only base model (longest job)
      f"{E} --model {MODEL} --setups A-Ctx --tag v04_actx {LIM_A}"]       #        then the no-gateway object-list arm on 400
Q1 = [f"{E} --model {MODEL} --setups B-Ctx-Feedback B-Ctx-Stop --tag v04_bctx {LIM}"]   # GPU 1: gateway-only baselines
t0 = time.time()
p0 = launch(0, Q0, "par_gpu0"); time.sleep(90)       # stagger so both processes do not compile into Unsloth's cache at the same moment
p1 = launch(1, Q1, "par_gpu1")
while p0.poll() is None or p1.poll() is None:
    time.sleep(300)
    print(f"[{(time.time()-t0)/60:5.0f} min] gpu0: {last_progress('par_gpu0')}\n            gpu1: {last_progress('par_gpu1')}", flush=True)
print(f"finished in {(time.time()-t0)/3600:.2f} h | exit codes: gpu0={p0.returncode} gpu1={p1.returncode}")
for n, p in (("par_gpu0", p0), ("par_gpu1", p1)):
    if p.returncode: print(f"\n--- {n} FAILED, last lines of its log ---\n" + open(f"{ROOT}/logs/{n}.log", errors="ignore").read()[-2500:])
assert p0.returncode == 0 and p1.returncode == 0, "a queue failed: see the log tail above (finished arms are already saved in results/)"


In [ ]:
# ---- silent-error check, then the merged analysis ----
for f in glob.glob("results/*.json"):
    r = json.load(open(f)); assert json.dumps(r["logs"]).count("<error") == 0, f"silent generation errors in {f}: lower EVAL_BATCH and rerun that arm"
print("no silent generation errors\n")
print(run("python rescore.py results data/test_episodes.json", "rescore"))


In [ ]:
# ---- save: results zip in /kaggle/working and (optional) private HF backup ----
shutil.make_archive(f"{ROOT}/permbench_results_v04", "zip", PROJ, "results")
print("saved", f"{ROOT}/permbench_results_v04.zip  (download it from the Output tab)")
if HF_REPO:
    from huggingface_hub import HfApi
    try:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: tok = os.environ.get("HF_TOKEN")
    api = HfApi(token=tok); api.upload_folder(folder_path="results", repo_id=HF_REPO, path_in_repo="results_v04", repo_type="model"); print("results backed up to", HF_REPO)
